# 视频特征表：检查与清洗


In [31]:
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """从当前目录向上定位作品集根目录。"""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Python").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请从 KuaiRand_Pure 目录或其子目录运行。")

print(PROJECT_ROOT)
PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MYSQL_IMPORT_DIR = PROJECT_ROOT / "data" / "mysql_import"


E:\Users\yanyan\Desktop\KuaiRand_Pure


## 1. 导入视频特征表


In [2]:
import pandas as pd

video_features = pd.read_csv(RAW_DIR / "video_features_basic_pure.csv")
video_features.head()


,video_id,author_id,video_type,upload_dt,upload_type,visible_status,video_duration,server_width,server_height,music_id,music_type,tag
0,0,7349781,NORMAL,2022-04-10,LongImport,0.0,87433.0,720.0,1280.0,9155697141,9.0,39
1,1,2103883,NORMAL,2022-04-10,Kmovie,0.0,218066.0,720.0,1280.0,6355810746,9.0,2
2,2,5067285,NORMAL,2022-04-09,ShortImport,0.0,9233.0,720.0,1280.0,6618412736,4.0,1
3,3,7048760,NORMAL,2022-04-11,Web,0.0,16433.0,720.0,1280.0,9161677205,9.0,7
4,4,8635271,NORMAL,2022-04-09,Web,0.0,38766.0,720.0,1280.0,9141092381,9.0,9


## 2. 对视频特征表进行数据质量检查


### 2.1 检查表格的基本结构，主键是否唯一，有无缺失值和重复值

In [3]:
#检查表格的基本结构
print("视频特征表形状：", video_features.shape)
print("视频特征表的索引",video_features.index)
print("视频特征表的字段名",video_features.columns.tolist())
print(video_features.info())
video_features.head()



视频特征表形状： (7583, 12)
视频特征表的索引 RangeIndex(start=0, stop=7583, step=1)
视频特征表的字段名 ['video_id', 'author_id', 'video_type', 'upload_dt', 'upload_type', 'visible_status', 'video_duration', 'server_width', 'server_height', 'music_id', 'music_type', 'tag']
<class 'pandas.DataFrame'>
RangeIndex: 7583 entries, 0 to 7582
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   video_id        7583 non-null   int64  
 1   author_id       7583 non-null   int64  
 2   video_type      7583 non-null   str    
 3   upload_dt       7583 non-null   str    
 4   upload_type     7583 non-null   str    
 5   visible_status  7583 non-null   float64
 6   video_duration  7344 non-null   float64
 7   server_width    7583 non-null   float64
 8   server_height   7583 non-null   float64
 9   music_id        7583 non-null   int64  
 10  music_type      7380 non-null   float64
 11  tag             7487 non-null   str    
dtypes: float64(5), int64(

,video_id,author_id,video_type,upload_dt,upload_type,visible_status,video_duration,server_width,server_height,music_id,music_type,tag
0,0,7349781,NORMAL,2022-04-10,LongImport,0.0,87433.0,720.0,1280.0,9155697141,9.0,39
1,1,2103883,NORMAL,2022-04-10,Kmovie,0.0,218066.0,720.0,1280.0,6355810746,9.0,2
2,2,5067285,NORMAL,2022-04-09,ShortImport,0.0,9233.0,720.0,1280.0,6618412736,4.0,1
3,3,7048760,NORMAL,2022-04-11,Web,0.0,16433.0,720.0,1280.0,9161677205,9.0,7
4,4,8635271,NORMAL,2022-04-09,Web,0.0,38766.0,720.0,1280.0,9141092381,9.0,9


In [8]:
#检查视频特征表的主键是否唯一
print(
    "video_id重复数量：",
    video_features["video_id"].duplicated().sum()
)
if video_features["video_id"].duplicated().sum()==0:
    print("表格的主键是唯一的")

video_id重复数量： 0
表格的主键是唯一的


In [9]:
#检查整行重复
print(
    "整行完全重复数量：",
    video_features.duplicated().sum()
)

整行完全重复数量： 0


In [10]:
#检查缺失值
missing_count = video_features.isna().sum()
print("各个字段的缺失值个数：",missing_count)

print("-"*60)
missing_rate = (
    video_features.isna().mean() * 100
).round(2)
print("各个字段的缺失值个数占总行数的比例",missing_rate)


print("-"*60)
missing_summary = pd.DataFrame({
    "缺失数量": missing_count,
    "缺失率(%)": missing_rate
})

missing_summary[
    missing_summary["缺失数量"] > 0
].sort_values(
    "缺失数量",
    ascending=False
)


各个字段的缺失值个数： video_id            0
author_id           0
video_type          0
upload_dt           0
upload_type         0
visible_status      0
video_duration    239
server_width        0
server_height       0
music_id            0
music_type        203
tag                96
dtype: int64
------------------------------------------------------------
各个字段的缺失值个数占总行数的比例 video_id          0.00
author_id         0.00
video_type        0.00
upload_dt         0.00
upload_type       0.00
visible_status    0.00
video_duration    3.15
server_width      0.00
server_height     0.00
music_id          0.00
music_type        2.68
tag               1.27
dtype: float64
------------------------------------------------------------


,缺失数量,缺失率(%)
video_duration,239,3.15
music_type,203,2.68
tag,96,1.27


### 2.2 检查表格的单个字段有效性

In [17]:
#检查枚举型字段，一般是有特定业务含义的字段。value_counts(dropna=False) 会把缺失值也统计出来。
category_cols = [
    "video_type",
    "upload_type",
    "visible_status",
    "music_type"
]

for col in category_cols:
    print(f"\n===== {col} =====")
    print(
        video_features[col]
        .value_counts(dropna=False)
    )



===== video_type =====
video_type
NORMAL     7506
AD           76
UNKNOWN       1
Name: count, dtype: int64

===== upload_type =====
upload_type
LongImport           2925
Web                  2416
ShortImport           917
Kmovie                879
LongPicture           143
UNKNOWN                80
PictureSet             73
LongCamera             64
ShortCamera            43
ShareFromOtherApp      29
FollowShoot            10
AiCutVideo              2
PhotoCopy               1
LipsSync                1
Name: count, dtype: int64

===== visible_status =====
visible_status
0.0    7583
Name: count, dtype: int64

===== music_type =====
music_type
9.0     6665
4.0      657
NaN      203
8.0       41
7.0       14
11.0       3
Name: count, dtype: int64
110
tag
39          815
3           651
9           418
6           378
20          344
           ... 
62,42,4       1
27,54         1
27,42,54      1
11,60         1
12,54         1
Name: count, Length: 110, dtype: int64


In [19]:
#检查数值字段的取值范围
number_cols = [
    "video_duration",
    "server_width",
    "server_height"
]

print("时长和分辨率基本统计：")
print(
    video_features[number_cols]
    .describe()
    .T
)

#视频时长、宽度和高度不应小于等于0。
print("\n零值数量：")
print(
    (video_features[number_cols] == 0)
    .sum()
)

print("\n负数数量：")
print(
    (video_features[number_cols] < 0)
    .sum()
)

#ID字段的0不一定是脏数据，因为这是公开脱敏数据，不能仅凭数值大小删除ID。
print("\nID字段中的0值数量（只观察，不直接删除）：")
print(
    (video_features[
        ["video_id", "author_id", "music_id"]
    ] == 0).sum()
)


时长和分辨率基本统计：
                 count           mean            std     min      25%  \
video_duration  7344.0  108615.889978  105497.991609  5000.0  31949.5   
server_width    7583.0     850.029804     232.489241   270.0    720.0   
server_height   7583.0    1140.159304     250.230933   448.0    960.0   

                    50%       75%        max  
video_duration  81170.5  148228.5  1177720.0  
server_width      720.0     960.0     2400.0  
server_height    1280.0    1280.0     2400.0  

零值数量：
video_duration    0
server_width      0
server_height     0
dtype: int64

负数数量：
video_duration    0
server_width      0
server_height     0
dtype: int64

ID字段中的0值数量（只观察，不直接删除）：
video_id       1
author_id      1
music_id     204
dtype: int64


In [14]:
#转换日期，将日期由字符串类型转换为datatime类型，进一步对datatime类型的数据按时间先后比较，再得到表格的时间范围。
upload_date_test = pd.to_datetime(
    video_features["upload_dt"],
    errors="coerce"
)

print(
    "上传日期解析失败数量：",
    upload_date_test.isna().sum()
)

print(
    "上传日期范围：",
    upload_date_test.min(),
    "至",
    upload_date_test.max()
)

上传日期解析失败数量： 0
上传日期范围： 2022-04-09 00:00:00 至 2022-04-11 00:00:00


In [13]:
#检查tag字段，有没有缺失，有没有空字符串。
tag_as_string = video_features["tag"].astype("string")

print(
    "标签缺失数量：",
    tag_as_string.isna().sum()
)

print(
    "标签空字符串数量：",
    tag_as_string.str.strip()
    .eq("")
    .fillna(False)
    .sum()
)
#原始数据：       ["39,68", "   ", <NA>]
#strip后：        ["39,68", "", <NA>]
#eq("")后：       [False, True, <NA>]
#fillna(False)后：[False, True, False]
#sum()结果：      1

print("\n出现次数最多的20种标签组合：")
print(
    video_features["tag"]
    .value_counts(dropna=False)
    .head(20)
)


上传日期解析失败数量： 0
上传日期范围： 2022-04-09 00:00:00 至 2022-04-11 00:00:00
标签缺失数量： 96
标签空字符串数量： 0

出现次数最多的20种标签组合：
tag
39       815
3        651
9        418
6        378
20       344
7        340
39,68    328
12       307
39,43    293
2        289
15       242
17       220
1        213
12,62    193
11       174
28       145
20,43    135
8        135
25       128
20,68     98
Name: count, dtype: int64


# 清洗数据

根据前面的检查结果，采用保守清洗：

1. 创建 video_features_clean，不直接修改原始 video_features；
2. 将上传日期转换为日期类型；
3. 保留原始毫秒时长，增加时长缺失标记，并由原始毫秒时长直接生成秒字段；
4. 音乐类型是类别编码，转换为 string，并将缺失统一记为 UNKNOWN；
5. 标签缺失填为字符串 UNKNOWN，方便后续分组，但不推测其真实内容；
6. video_type 和 upload_type 原有的 UNKNOWN 作为明确的未知类别保留；
7. visible_status 全部相同，保留字段，但后续不用于用户或内容分层。


In [23]:
rows_before = video_features.shape[0]

#1. 创建 video_features_clean，不直接修改原始 video_features；
video_features_clean = video_features.copy()

#创建字段 upload_date_clean
video_features_clean["upload_date_clean"] = pd.to_datetime(
    video_features_clean["upload_dt"],
    errors="coerce"
)

video_features_clean["is_duration_missing"] = (
    video_features_clean["video_duration"].isna() ).astype("int8")

video_features_clean["video_duration_seconds"] = (
    video_features_clean["video_duration"] / 1000
)


video_features_clean["music_type_clean"] = (
    video_features_clean["music_type"]
    .astype("Int16")
    .astype("string")
    .fillna("UNKNOWN")
)

video_features_clean["tag_clean"] = (
    video_features_clean["tag"]
    .astype("string")
    .fillna("UNKNOWN")
)

rows_after = video_features_clean.shape[0]

print("清洗前行数：", rows_before)
print("清洗后行数：", rows_after)
print("删除完全重复行数：", rows_before - rows_after)


清洗前行数： 7583
清洗后行数： 7583
删除完全重复行数： 0


# 验证清洗结果

本节验证：

- 行数是否意外减少；
- video_id 是否仍然唯一；
- 日期是否全部成功转换；
- 时长缺失标记是否与清洗后时长缺失数量一致；
- 标签清洗后是否还有缺失；
- 原始缺失信息是否得到保留，而不是被虚构数值覆盖。


In [26]:
#video_id 是否仍然唯一
print(
    "video_id重复数量：",
    video_features_clean["video_id"]
    .duplicated()
    .sum()
)

print(
    "上传日期转换失败数量：",
    video_features_clean["upload_date_clean"]
    .isna()
    .sum()
)

print(
    "时长缺失标记数量：",
    video_features_clean["is_duration_missing"]
    .sum()
)


print(
    "清洗后音乐类型的缺失数量：",
    video_features_clean["music_type_clean"]
    .isna()
    .sum()
)

print(
    "清洗后音乐类型UNKNOWN数量：",
    video_features_clean["music_type_clean"]
    .eq("UNKNOWN")
    .sum()
)

print(
    "清洗后标签的缺失数量：",
    video_features_clean["tag_clean"]
    .isna()
    .sum()
)

print(
    "清洗后标签UNKNOWN数量：",
    video_features_clean["tag_clean"]
    .eq("UNKNOWN")
    .sum()
)

print("\n新增清洗字段的数据类型：")
print(
    video_features_clean[[
        "upload_date_clean",
        "is_duration_missing",
        "video_duration_seconds",
        "music_type_clean",
        "tag_clean"
    ]].dtypes
)


video_id重复数量： 0
上传日期转换失败数量： 0
时长缺失标记数量： 239
清洗后音乐类型的缺失数量： 0
清洗后音乐类型UNKNOWN数量： 203
清洗后标签的缺失数量： 0
清洗后标签UNKNOWN数量： 96

新增清洗字段的数据类型：
upload_date_clean         datetime64[us]
is_duration_missing                 int8
video_duration_seconds           float64
music_type_clean                  string
tag_clean                         string
dtype: object


## 导出视频特征清洗表

In [28]:
processed_dir = PROCESSED_DIR
processed_dir.mkdir(parents=True, exist_ok=True)

In [29]:
video_path = (
    processed_dir
    / "video_features_basic_clean.parquet"
)

video_features_clean.to_parquet(
    video_path,
    index=False
)

print(
    "视频基础表已导出：",
    video_path,
    video_features_clean.shape
)


视频基础表已导出： E:\Users\yanyan\Desktop\KuaiRand_Pure\data\processed\video_features_basic_clean.parquet (7583, 17)
